# iGSM-med eval — per-slice correct / mistake analysis

Pick a **slice** (e.g. `med_pq_op_le15`, `med_pq_op_eq20`, `med_pq_reask`), then this notebook lists
the problems the model **solved** and the ones it **did not**, decoded as text. After each section,
set `PICK` to one of the displayed `#` indices and re-run the graph cell to draw that problem's
dependency graph (the iGSM `whole_template`, with the solution subgraph `template` highlighted).

> Requires an eval log from `uv run python -m src.eval.run ...` (the newest `*.json` in `LOG_DIR` is
> used). Graphs are cleanest for **small-op** problems (`op_le15`, `op_eq15`); op=20+ graphs are dense.

In [ ]:
# === Configure here ===
SLICE = "med_pq_op_le15"                       # e.g. "med_pq_op_le15", "med_pq_op_eq20", "med_pq_reask"
LOG_DIR = "logs/eval"                   # dir with inspect_ai *.json logs (newest is used)
N_SHOW = 10                        # how many examples to list per section
# ======================

from pathlib import Path
import base64
import json
import pickle
import textwrap

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import Markdown, display
from transformers import GPT2TokenizerFast

from src.data.igsm import ensure_igsm_submodule

ensure_igsm_submodule()
_TOK = GPT2TokenizerFast.from_pretrained("gpt2")
_ROOT = (-1, 0, 0, 0)  # iGSM synthetic root node; dropped when drawing


def load_samples(log_dir):
    """Read the newest eval log in `log_dir`; return (samples, log_filename)."""
    logs = sorted(Path(log_dir).glob("*.json"))
    assert logs, f"no eval logs in {log_dir!r} (run `python -m src.eval.run` first)"
    return json.loads(logs[-1].read_text())["samples"], logs[-1].name


def decode_problem(sample):
    """Unpickle the gold iGSM `Problem` carried (base64) in the sample metadata."""
    return pickle.loads(base64.b64decode(sample["metadata"]["problem"]))


def verdict(sample):
    """True if the model's generation scored correct (iGSM true_correct)."""
    return bool(sample["scores"]["igsm_correctness"]["value"])


def to_text(ids):
    """Decode token ids to readable text, stripping the [50256, 222, ..., 223] wrapper."""
    ids = list(ids)
    if len(ids) >= 2 and ids[0] == 50256 and ids[1] == 222:
        ids = ids[2:]
    if ids and ids[-1] == 223:
        ids = ids[:-1]
    return _TOK.decode(ids, skip_special_tokens=True)


def show_problem(sample, index):
    md = sample["metadata"]
    gen = (sample.get("output") or {}).get("completion") or "(empty)"
    display(Markdown(
        f"**#{index}** · slice `{md['slice']}` · op={md['op']} · "
        f"gold_answer={md['gold_answer']} · {'✅ solved' if verdict(sample) else '❌ wrong'}\n\n"
        f"**Problem:** {to_text(md['prompt_ids'])}\n\n"
        f"**Generation:** {gen}"
    ))


def draw_problem_graph(sample, title=None):
    """Draw whole_template (grey) with the solution subgraph `template` (orange) on top."""
    prob = decode_problem(sample)
    whole = prob.whole_template
    tpl_nodes = set(prob.template.nodes())
    keep = [n for n in whole.nodes() if n != _ROOT]
    g = whole.subgraph(keep).copy()
    labels = {n: "\n".join(textwrap.wrap(prob.get_param(n), 16)) for n in keep}
    pos = nx.spring_layout(g, seed=0)  # numpy-only (scipy not installed; kamada needs it)
    in_tpl = [n for n in keep if n in tpl_nodes]
    out_tpl = [n for n in keep if n not in tpl_nodes]
    fig, ax = plt.subplots(figsize=(10, 7))
    nx.draw_networkx_edges(g, pos, ax=ax, edge_color="#bbbbbb", arrowsize=12)
    if out_tpl:
        nx.draw_networkx_nodes(g, pos, nodelist=out_tpl, ax=ax, node_color="#dcdcdc", node_size=400)
    nx.draw_networkx_nodes(g, pos, nodelist=in_tpl, ax=ax, node_color="#f0a040",
                           node_size=650, edgecolors="#b07020")
    nx.draw_networkx_labels(g, pos, labels=labels, ax=ax, font_size=6)
    ax.legend(handles=[
        mpatches.Patch(color="#f0a040", label="needed for the answer (template)"),
        mpatches.Patch(color="#dcdcdc", label="rest of the world model"),
    ], loc="lower center", ncol=2)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


print("ready")

In [ ]:
samples, log_name = load_samples(LOG_DIR)
in_slice = [s for s in samples if s["metadata"]["slice"] == SLICE]
correct = [s for s in in_slice if verdict(s)]
mistakes = [s for s in in_slice if not verdict(s)]
print(f"log:     {log_name}")
print(f"slice:   {SLICE}")
print(f"total:   {len(in_slice)}   solved: {len(correct)}   not solved: {len(mistakes)}")
assert in_slice, f"no samples for slice {SLICE!r} in {log_name}"

## ✅ Solved by the model
The generation was fully correct: right final answer **and** every calculation (mod 23) **and**
every parameter-dependency edge (iGSM `true_correct`).

In [ ]:
for i, s in enumerate(correct[:N_SHOW]):
    show_problem(s, i)
print(f"(showing {min(len(correct), N_SHOW)} of {len(correct)} solved)")

### Visualize a solved problem's graph
Set `PICK` to one of the `#` indices above, then re-run this cell.

In [ ]:
PICK = 0
if not correct:
    print("no solved problems in this slice/log")
elif PICK >= len(correct):
    print(f"PICK={PICK} out of range (only {len(correct)} solved)")
else:
    draw_problem_graph(correct[PICK], title=f"solved #{PICK} · {SLICE}")

## ❌ NOT solved by the model
The generation was wrong somewhere — final answer, a calculation, or a dependency edge.

In [ ]:
for i, s in enumerate(mistakes[:N_SHOW]):
    show_problem(s, i)
print(f"(showing {min(len(mistakes), N_SHOW)} of {len(mistakes)} not solved)")

### Visualize an unsolved problem's graph
Set `PICK` to one of the `#` indices above, then re-run this cell.

In [ ]:
PICK = 0
if not mistakes:
    print("no unsolved problems in this slice/log")
elif PICK >= len(mistakes):
    print(f"PICK={PICK} out of range (only {len(mistakes)} not solved)")
else:
    draw_problem_graph(mistakes[PICK], title=f"unsolved #{PICK} · {SLICE}")